# Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)

# Funkcije

In [ ]:
def aktivacija(z, ime, odvod=False):
    """Aktivacijske funkcije in njihovi odvodi."""
    z = np.asarray(z, float)
    # TODO: "tanh", "sigmoid", "relu", "linear"
    pass
    raise ValueError(ime)


def ustvari_mrezo(arhitektura, rng=rng):
    """Seznam parov (W, b) za arhitekturo, npr. [2, 8, 1]. Uteži ~ N(0, 1/n_vhod) (Xavier)."""
    # TODO
    pass


def naprej(mreza, X, aktivacije):
    """Propagacija naprej za paket vhodov X (N, n_vhod). Vrne seznama z (pred aktivacijo) in a (po njej);
    a[0] = X, a[-1] je izhod mreže."""
    zs, As = [], [X]
    for (W, b), f in zip(mreza, aktivacije):
        # TODO: z = a W + b, a = f(z)
        pass
    return zs, As


def izguba(y, y_nap, ime="MSE"):
    """Izguba in njen odvod po izhodu mreže (povprečje po vzorcih)."""
    N = len(y)
    if ime == "MSE":
        # TODO
        pass
    if ime == "BCE":     # binarna navzkrižna entropija (izhod s sigmoido)
        p = np.clip(y_nap, 1e-12, 1 - 1e-12)
        return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)), (p - y) / (p * (1 - p)) / N
    raise ValueError(ime)


def nazaj(mreza, zs, As, dL_da, aktivacije):
    """Vzvratno razširjanje napake. Vrne seznam gradientov (dW, db) za vse plasti."""
    grad = [None] * len(mreza)
    delta = dL_da * aktivacija(zs[-1], aktivacije[-1], odvod=True)
    for l in reversed(range(len(mreza))):
        # TODO: dW = a_{l-1}^T delta, db = vsota delta po vzorcih; delta za prejšnjo plast
        pass
    return grad


def uci(mreza, X, y, aktivacije, eta=0.1, epohe=2000, ime_izgube="MSE", X_test=None, y_test=None):
    """Gradientni spust na celotnem paketu. Vrne zgodovino izgube (učna, testna)."""
    zg_u, zg_t = [], []
    for ep in range(epohe):
        zs, As = naprej(mreza, X, aktivacije)
        L, dL = izguba(y, As[-1], ime_izgube)
        grad = nazaj(mreza, zs, As, dL, aktivacije)
        # TODO: posodobi uteži in pragove (W -= eta dW, b -= eta db)
        pass
        zg_u.append(L)
        if X_test is not None:
            zg_t.append(izguba(y_test, naprej(mreza, X_test, aktivacije)[1][-1], ime_izgube)[0])
    return np.array(zg_u), np.array(zg_t)


def numericni_gradient(mreza, X, y, aktivacije, h=1e-6):
    """Gradient izgube MSE po vseh utežeh s centralnimi diferencami (za preverjanje)."""
    grad = []
    for W, b in mreza:
        gW, gb = np.zeros_like(W), np.zeros_like(b)
        for P, G in ((W, gW), (b, gb)):
            for idx in np.ndindex(P.shape):
                staro = P[idx]
                P[idx] = staro + h
                Lp = izguba(y, naprej(mreza, X, aktivacije)[1][-1])[0]
                P[idx] = staro - h
                Lm = izguba(y, naprej(mreza, X, aktivacije)[1][-1])[0]
                P[idx] = staro
                G[idx] = (Lp - Lm) / (2 * h)
        grad.append((gW, gb))
    return grad

# Testno območje

In [ ]:
# P1: vrednosti pri z = 0
print({f: (float(aktivacija(0.0, f)), float(aktivacija(0.0, f, True))) for f in ("tanh", "sigmoid", "relu", "linear")})

# P3: en nevron s sigmoido
mreza_p3 = [(np.array([[0.5], [-1.0]]), np.array([0.2]))]
zs, As = naprej(mreza_p3, np.array([[1.0, 2.0]]), ["sigmoid"])
L, dL = izguba(np.array([[1.0]]), As[-1])
g = nazaj(mreza_p3, zs, As, dL, ["sigmoid"])
print(f"P3: z = {zs[0][0, 0]:.3f}, a = {As[-1][0, 0]:.4f}, L = {L:.4f}, dL/dw = {g[0][0].ravel().round(4)}, dL/db = {g[0][1].round(4)}")

# P2: število parametrov
for arh in ([1, 10, 1], [2, 8, 1], [2, 2, 1]):
    print(arh, "parametrov:", sum(W.size + b.size for W, b in ustvari_mrezo(arh)))

# Naloga 1 - Slika 1

In [ ]:
z = np.linspace(-4, 4, 401)
fig = plt.figure(figsize=(11, 4))
ax1 = fig.add_subplot(121)
ax2 = fig.add_subplot(122)
# TODO: aktivacijske funkcije in odvodi; primerjava odvoda z numeričnim (np.gradient)
fig.tight_layout()
plt.show()

# Naloga 2

In [ ]:
# Preverjanje gradienta na majhni mreži [2, 3, 1]
X_g = rng.normal(size=(5, 2))
y_g = rng.normal(size=(5, 1))
akt_g = ["tanh", "linear"]
mreza_g = ustvari_mrezo([2, 3, 1])
# TODO: gradient z vzvratnim razširjanjem in numerični gradient; relativna napaka za vsako plast (Preglednica 2)

# Naloga 3 - Slika 2

In [ ]:
# Regresija: y = sin(2 pi x) + šum
x_u = rng.uniform(0, 1, (40, 1))
y_u = np.sin(2 * np.pi * x_u) + rng.normal(0, 0.1, x_u.shape)
x_t = np.linspace(0, 1, 200)[:, None]
y_t = np.sin(2 * np.pi * x_t)
akt = ["tanh", "linear"]
fig = plt.figure(figsize=(12, 4))
ax1 = fig.add_subplot(121)
ax2 = fig.add_subplot(122)
# TODO: učenje mreže [1, 10, 1] pri eta = 0.01, 0.05, 0.2; krivulje izgube in napoved (Preglednica 3)
fig.tight_layout()
plt.show()

In [ ]:
# Preglednica 3 (nadaljevanje): vpliv števila skritih nevronov (eta = 0.05)
# TODO

# Naloga 4 - Slika 3

In [ ]:
# Klasifikacija: dva koncentrična obroča
N = 200
r = np.r_[rng.normal(1.0, 0.15, N), rng.normal(2.0, 0.15, N)]
fi = rng.uniform(0, 2 * np.pi, 2 * N)
X_k = np.column_stack([r * np.cos(fi), r * np.sin(fi)])
y_k = np.r_[np.zeros(N), np.ones(N)][:, None]
akt_k = ["tanh", "sigmoid"]
xx, yy = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
mreza_tocke = np.column_stack([xx.ravel(), yy.ravel()])
fig = plt.figure(figsize=(16, 4))
# TODO: mreže [2, n, 1] za n = 1, 2, 4, 8 z izgubo BCE; točnost in odločitvena meja (Preglednica 4)
fig.tight_layout()
plt.show()